# 03. 문화결핍지수 산출

문화공급지수·수요지수·접근성 패널티·미스매치 패널티를 결합해
시군구별 Culture Gap Score를 산출합니다.

$$\text{Culture Gap Score} = 0.4 \times \text{Demand} + 0.3 \times \text{Supply Deficit} + 0.2 \times \text{Access Penalty} + 0.1 \times \text{Mismatch}$$

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
from preprocessing import load_raw_data
from feature_engineering import build_region_features_from_raw
from scoring import add_scores
raw = load_raw_data()
features = build_region_features_from_raw(raw)
scored = add_scores(features)
print('점수 통계:')
scored[['supply_score','demand_score','access_penalty','mismatch_penalty','culture_gap_score']].describe().round(1)

## 3-1. 위험 등급 분포

In [ ]:
risk_cnt = scored['risk_level'].value_counts()
colors = {'고위험':'#c84c3a','주의':'#e8956b','보통':'#f5c842','양호':'#4caf7d'}
risk_cnt.plot.pie(colors=[colors.get(r,'gray') for r in risk_cnt.index],
                  autopct='%1.1f%%', figsize=(5,5))
plt.title('전국 시군구 위험 등급 비율')
plt.ylabel('')
plt.show()
print(risk_cnt)

## 3-2. 점수 구성 요소 분석

In [ ]:
top20 = scored.sort_values('culture_gap_score', ascending=False).head(20)
components = top20[['region_name','demand_score','supply_score',
                      'access_penalty','mismatch_penalty','culture_gap_score']]
components

## 3-3. 시도별 평균 점수

In [ ]:
prov_avg = scored.groupby('province')['culture_gap_score'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(prov_avg.index, prov_avg.values,
               color=['#c84c3a' if s >= 60 else '#e8956b' if s >= 55 else '#f5c842' for s in prov_avg.values])
ax.axvline(x=60, color='red', linestyle='--', linewidth=0.8)
ax.set_xlabel('평균 문화결핍지수')
ax.set_title('시도별 평균 문화결핍지수')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3-4. 문화결핍지수 하위 지역 (가장 양호)

In [ ]:
scored.sort_values('culture_gap_score').head(10)
[['region_name','province','supply_score','demand_score','culture_gap_score','risk_level']]